Train a Perceiver-style compressor that maps T5 encoder hidden states from 256 tokens to 128 tokens while preserving frozen decoder outputs.

In [11]:
!pip install transformers datasets accelerate sentencepiece

In [12]:
!pip uninstall -y transformers datasets huggingface_hub
!pip install transformers==4.52.4 datasets==3.6.0 huggingface_hub==0.33.5

Found existing installation: transformers 5.10.1
Uninstalling transformers-5.10.1:
  Successfully uninstalled transformers-5.10.1
Found existing installation: datasets 4.0.0
Uninstalling datasets-4.0.0:
  Successfully uninstalled datasets-4.0.0
Found existing installation: huggingface_hub 1.17.0
Uninstalling huggingface_hub-1.17.0:
  Successfully uninstalled huggingface_hub-1.17.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 39.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.7/515.7 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 30.9 MB/s eta 0:00:00
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is th

In [1]:
import transformers
import datasets
import huggingface_hub

print(transformers.__version__)
print(datasets.__version__)
print(huggingface_hub.__version__)

4.52.4
3.6.0
0.33.5


In [2]:
from datasets import load_dataset

dataset = load_dataset("ag_news")
print(dataset)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})


In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from datasets import load_dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
)

from transformers.modeling_outputs import (
    BaseModelOutput,
)

In [4]:
MODEL_NAME = "google/flan-t5-base"

device = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME
).to(device)

model.eval()

T5ForConditionalGeneration(
  (shared): Embedding(32128, 768)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 768)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=768, out_features=768, bias=False)
              (k): Linear(in_features=768, out_features=768, bias=False)
              (v): Linear(in_features=768, out_features=768, bias=False)
              (o): Linear(in_features=768, out_features=768, bias=False)
              (relative_attention_bias): Embedding(32, 12)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseGatedActDense(
              (wi_0): Linear(in_features=768, out_features=2048, bias=False)
              (wi_1): Linear(in_features=768, out_features=2048, bias=False)
              (wo):

In [5]:
for p in model.parameters():
    p.requires_grad = False

In [6]:
class PerceiverCompressor(
    nn.Module
):
    def __init__(
        self,
        hidden_size=768,
        num_latents=128,
        heads=8,
    ):
        super().__init__()

        self.latents = nn.Parameter(
            torch.randn(
                num_latents,
                hidden_size
            )
        )

        self.cross_attn = (
            nn.MultiheadAttention(
                hidden_size,
                heads,
                batch_first=True,
            )
        )

        self.norm = nn.LayerNorm(
            hidden_size
        )

    def forward(
        self,
        hidden_states,
    ):
        B = hidden_states.size(0)

        latents = (
            self.latents
            .unsqueeze(0)
            .expand(B,-1,-1)
        )

        compressed,_ = (
            self.cross_attn(
                latents,
                hidden_states,
                hidden_states,
            )
        )

        return self.norm(
            compressed
        )

In [7]:
compressor = (
    PerceiverCompressor()
    .to(device)
)

optimizer = torch.optim.AdamW(
    compressor.parameters(),
    lr=1e-4,
)

In [8]:
dataset = load_dataset(
      "ag_news"
)

train_texts = []

for x in dataset["train"]:
    txt = x["text"]

    if len(txt) > 20:
        train_texts.append(txt)

print(
    len(train_texts)
)

120000


In [9]:
def training_step(text):

    inputs = tokenizer(
        text,
        max_length=256,
        truncation=True,
        padding="max_length",
        return_tensors="pt",
    )

    inputs = {
        k: v.to(device)
        for k, v in inputs.items()
    }

    with torch.no_grad():

        enc = model.encoder(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            return_dict=True,
        )

    hidden = enc.last_hidden_state

    compressed = compressor(
        hidden
    )
    if torch.rand(1).item() < 0.0005:
      print(
        "hidden:",
        hidden.shape,
        "compressed:",
        compressed.shape,
    )


    decoder_input_ids = model._shift_right(
        inputs["input_ids"]
    )

    with torch.no_grad():

        teacher_outputs = model.decoder(
            input_ids=decoder_input_ids,
            encoder_hidden_states=hidden,
            encoder_attention_mask=inputs[
                "attention_mask"
            ],
            output_hidden_states=True,
            return_dict=True,
        )

    student_outputs = model.decoder(
        input_ids=decoder_input_ids,
        encoder_hidden_states=compressed,
        encoder_attention_mask=torch.ones(
            compressed.shape[:2],
            device=device,
            dtype=torch.long,
        ),
        output_hidden_states=True,
        return_dict=True,
    )

    teacher_hidden = (
        teacher_outputs.last_hidden_state
    )

    student_hidden = (
        student_outputs.last_hidden_state
    )

    loss = F.mse_loss(
        student_hidden,
        teacher_hidden,
    )

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

    return loss.item()

In [13]:
compressor = PerceiverCompressor().to(device)

compressor.load_state_dict(
    torch.load(
        "compressor_256_to_128.pt",
        map_location=device,
    )
)

compressor.eval()

print("Checkpoint loaded.")

Checkpoint loaded.


In [14]:
for epoch in range(5):

    total = 0

    for i,text in enumerate(
        train_texts[:5000]
    ):

        loss = training_step(
            text
        )

        total += loss

        if i % 100 == 0:

            print(
                epoch,
                i,
                total/(i+1)
            )

Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


0 0 0.017837032675743103


KeyboardInterrupt: 

In [18]:
from transformers.modeling_outputs import BaseModelOutput

text = """
Encoder hidden states are reusable semantic computation artifacts.
"""

inputs = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    padding="max_length",
    max_length=256,
)

inputs = {
    k: v.to(device)
    for k, v in inputs.items()
}

with torch.no_grad():

    teacher_output = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=50,
    )

print(
    "Teacher:",
    tokenizer.decode(
        teacher_output[0],
        skip_special_tokens=True,
    )
)

Teacher: Encoder hidden states are reusable semantic computation artifacts.


In [19]:
with torch.no_grad():

    encoder_outputs = model.encoder(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        return_dict=True,
    )

    hidden = encoder_outputs.last_hidden_state

    compressed = compressor(hidden)

    compressed_mask = torch.ones(
        compressed.shape[:2],
        dtype=torch.long,
        device=device,
    )

    student_output = model.generate(
        encoder_outputs=BaseModelOutput(
            last_hidden_state=compressed
        ),
        attention_mask=compressed_mask,
        max_new_tokens=50,
    )

print(
    "Student:",
    tokenizer.decode(
        student_output[0],
        skip_special_tokens=True,
    )
)

Student:                                                  


In [20]:
examples = [
    train_texts[0],
    train_texts[10],
    train_texts[20],
    train_texts[30],
    train_texts[40],
]

for idx, text in enumerate(examples):

    print("\n" + "=" * 80)
    print("EXAMPLE", idx)
    print("=" * 80)

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding="max_length",
        max_length=256,
    )

    inputs = {
        k: v.to(device)
        for k, v in inputs.items()
    }

    with torch.no_grad():

        teacher = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=40,
        )

        enc = model.encoder(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            return_dict=True,
        )

        compressed = compressor(
            enc.last_hidden_state
        )

        student = model.generate(
            encoder_outputs=BaseModelOutput(
                last_hidden_state=compressed
            ),
            attention_mask=torch.ones(
                compressed.shape[:2],
                dtype=torch.long,
                device=device,
            ),
            max_new_tokens=40,
        )

    print(
        "Teacher:",
        tokenizer.decode(
            teacher[0],
            skip_special_tokens=True,
        )
    )

    print()

    print(
        "Student:",
        tokenizer.decode(
            student[0],
            skip_special_tokens=True,
        )
    )


EXAMPLE 0
Teacher: Short-sellers are seeing green again, as the market is reversing its downward trend.

Student:                                        

EXAMPLE 1
Teacher: Oil and Economy Cloud Stocks' Outlook

Student:                                        

EXAMPLE 2
Teacher: Google's IPO is a flop, with the company's IPO a flop.

Student:                                        

EXAMPLE 3
Teacher: Japan's nuclear plant company is to close its reactors for safety checks after a fatal accident at the plant in the north of the country.

Student:                                        

EXAMPLE 4
Teacher: UZIs and AK-47s could again be flooding the streets.

Student:                                        


In [34]:
with torch.no_grad():

    enc = model.encoder(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        return_dict=True,
    )

    hidden = enc.last_hidden_state

    compressed = compressor(hidden)

    decoder_input_ids = model._shift_right(
        inputs["input_ids"]
    )

    teacher_outputs = model.decoder(
        input_ids=decoder_input_ids,
        encoder_hidden_states=hidden,
        encoder_attention_mask=inputs["attention_mask"],
        return_dict=True,
    )

    student_outputs = model.decoder(
        input_ids=decoder_input_ids,
        encoder_hidden_states=compressed,
        encoder_attention_mask=torch.ones(
            compressed.shape[:2],
            device=device,
            dtype=torch.long,
        ),
        return_dict=True,
    )

mse = torch.mean(
    (
        teacher_outputs.last_hidden_state
        -
        student_outputs.last_hidden_state
    ) ** 2
)

print("MSE =", mse.item())

MSE = 0.020270073786377907


In [21]:
with torch.no_grad():

    enc = model.encoder(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        return_dict=True,
    )

    hidden = enc.last_hidden_state

    compressed = compressor(hidden)

    decoder_input_ids = model._shift_right(
        inputs["input_ids"]
    )

    teacher = model(
        encoder_outputs=BaseModelOutput(
            last_hidden_state=hidden
        ),
        decoder_input_ids=decoder_input_ids,
        return_dict=True,
    )

    student = model(
        encoder_outputs=BaseModelOutput(
            last_hidden_state=compressed
        ),
        decoder_input_ids=decoder_input_ids,
        return_dict=True,
    )

teacher_token = teacher.logits.argmax(-1)
student_token = student.logits.argmax(-1)

print(
    "teacher:",
    tokenizer.decode(
        teacher_token[0][:50]
    )
)

print(
    "student:",
    tokenizer.decode(
        student_token[0][:50]
    )
)

teacher: s</s> the Abrs of Legal weapons</s> can all share the outrage, expressed by columnist Steve Bailey ('SSummer Sizzler, quot; Aug. 11), at the killings in the city's
student:                                                  


In [22]:
with torch.no_grad():

    teacher = model(
        encoder_outputs=BaseModelOutput(
            last_hidden_state=hidden
        ),
        decoder_input_ids=decoder_input_ids,
        return_dict=True,
    )

    student = model(
        encoder_outputs=BaseModelOutput(
            last_hidden_state=compressed
        ),
        decoder_input_ids=decoder_input_ids,
        return_dict=True,
    )

teacher_tokens = teacher.logits.argmax(-1)
student_tokens = student.logits.argmax(-1)

print(
    tokenizer.decode(
        teacher_tokens[0]
    )
)

print(
    tokenizer.decode(
        student_tokens[0]
    )
)

s</s> the Abrs of Legal weapons</s> can all share the outrage, expressed by columnist Steve Bailey ('SSummer Sizzler, quot; Aug. 11), at the killings in the city's poor neighborhoods. But there's no need to share his ignorance. He argues that renewal of the so-called assault weapon ban, claiming that otherwise, ''UZIs and AK-47s could again be flooding the streets.'...;</s> letter...</s></s>                                                                                                                                                 
                                                                                                                                                                                                                                                               


In [23]:
teacher_top = teacher.logits.argmax(-1)
student_top = student.logits.argmax(-1)

agreement = (
    teacher_top ==
    student_top
).float().mean()

print(agreement.item())

0.58984375


In [24]:
teacher_first = teacher.logits[0,0].argmax()
student_first = student.logits[0,0].argmax()

print(
    teacher_first.item(),
    tokenizer.decode([teacher_first.item()])
)

print(
    student_first.item(),
    tokenizer.decode([student_first.item()])
)

3 
3 


In [25]:
teacher_top5 = torch.topk(
    teacher.logits,
    k=5,
    dim=-1
).indices

student_top1 = student.logits.argmax(
    dim=-1,
    keepdim=True
)

matches = (
    teacher_top5 ==
    student_top1
)

top5_agreement = (
    matches.any(dim=-1)
    .float()
    .mean()
)

print(
    "Top-5 agreement:",
    top5_agreement.item()
)

Top-5 agreement: 0.80859375


In [26]:
teacher_top5 = torch.topk(
    teacher.logits,
    k=5,
    dim=-1
).indices

student_top5 = torch.topk(
    student.logits,
    k=5,
    dim=-1
).indices

overlap = []

for i in range(
    teacher_top5.shape[1]
):

    t = set(
        teacher_top5[0, i]
        .cpu()
        .tolist()
    )

    s = set(
        student_top5[0, i]
        .cpu()
        .tolist()
    )

    overlap.append(
        len(
            t.intersection(s)
        ) / 5.0
    )

print(
    "Top-5 overlap:",
    sum(overlap) / len(overlap)
)

Top-5 overlap: 0.35000000000000003


In [27]:
logit_mse = torch.mean(
    (
        teacher.logits -
        student.logits
    ) ** 2
)

print(
    "Logit MSE:",
    logit_mse.item()
)

Logit MSE: 16.772064208984375


In [28]:
import torch.nn.functional as F

teacher_probs = F.softmax(
    teacher.logits,
    dim=-1
)

student_log_probs = F.log_softmax(
    student.logits,
    dim=-1
)

kl = F.kl_div(
    student_log_probs,
    teacher_probs,
    reduction="batchmean"
)

print(
    "KL divergence:",
    kl.item()
)

KL divergence: 1275.941650390625


In [29]:
teacher_max = teacher.logits.max().item()
teacher_min = teacher.logits.min().item()

student_max = student.logits.max().item()
student_min = student.logits.min().item()

print(
    teacher_min,
    teacher_max
)

print(
    student_min,
    student_max
)

-92.13641357421875 5.431277751922607
-64.04989624023438 -0.18083715438842773


In [30]:
teacher_entropy = (
    -(torch.softmax(
        teacher.logits,
        dim=-1
    ) *
      torch.log_softmax(
        teacher.logits,
        dim=-1
      )
    ).sum(-1)
).mean()

student_entropy = (
    -(torch.softmax(
        student.logits,
        dim=-1
    ) *
      torch.log_softmax(
        student.logits,
        dim=-1
      )
    ).sum(-1)
).mean()

print(
    teacher_entropy.item()
)

print(
    student_entropy.item()
)

2.79243803024292
2.353595018386841


In [31]:
print(
    teacher.logits.mean().item(),
    teacher.logits.std().item()
)

print(
    student.logits.mean().item(),
    student.logits.std().item()
)

-18.824487686157227 6.804194927215576
-20.45339012145996 5.772799015045166


In [35]:
teacher_norm = teacher_outputs.last_hidden_state.norm(
    dim=-1
).mean()

student_norm = student_outputs.last_hidden_state.norm(
    dim=-1
).mean()

print(
    teacher_norm.item()
)

print(
    student_norm.item()
)

10.406413078308105
9.813036918640137


In [36]:
import torch.nn.functional as F

cos = F.cosine_similarity(
    teacher_outputs.last_hidden_state.flatten(),
    student_outputs.last_hidden_state.flatten(),
    dim=0
)

print(cos.item())

0.9292904734611511
